# NB1 — LASI Feature Engineering (revised v2)

**What this notebook does:** turns the merged LASI file (`lasi_merged_clear.csv`) into a clean modelling table with explicit,
documented handling of every problem found in the review. It does **no imputation any more** (see change 3).

**Input:** `lasi_merged_clear.csv` (66,859 rows). **Outputs:**
- `lasi_model_ready.csv` — all features/outcomes, missing values left as genuine missing (NaN)
- `lasi_model_ready_complete_case.csv` — rows with no missing predictor and a known primary outcome (sensitivity analysis)

## What changed versus the previous version, and why

| # | Change | Why (what was wrong before) |
|---|---|---|
| 1 | **Tobacco is now 3 levels: Never / Former / Current** (+ a small genuine Unknown). | The old rule labelled only "never users" as `No`, so ~3,400 **former** users (who answered "Quit smoking"/"Quit smokeless") fell into "Missing". Former users had a much higher disease prevalence (they plausibly quit *after* diagnosis), so the "Missing" level became the model's top predictor and "current tobacco" looked protective. That is a coding artefact, not a finding. |
| 2 | **Family-history "unknown" stays missing.** | The old code did `(sum > 0).astype(float)`; `NaN > 0` is `False`, so "don't know"/refused answers (e.g. 207 in `fm305`) were silently turned into "no family history". |
| 3 | **Imputation removed from this notebook.** | The old `IterativeImputer` was fit on all rows *before* the train/test split, so test rows influenced the imputation model (a mild form of leakage). Imputation is now a step inside the model pipeline (NB3), fit on training folds only. |
| 4 | **Range checks for BP and pulse** (not just BMI). | The raw file contains impossible zeros for systolic/diastolic BP and pulse. They were previously passed to the model. |
| 5 | **`household_id` and `state_code` created.** | Used for a household-grouped split (NB2) and leave-states-out validation (NB4). |
| 6 | **Outcome definitions made explicit**; new secondary outcome `ihd_composite` (heart attack OR coronary disease). | `ht006` is *any* self-reported chronic heart disease (it can include heart failure, valvular/rheumatic disease, etc.) — it is **not** a coronary-artery-disease label. |
| 7 | `current_smoking_only` added. | The WHO chart is defined for smoking, and LASI's smokeless tobacco is very common; NB5 needs both definitions for a sensitivity analysis. |

## Assumptions (stated explicitly)
- **Household:** `household_id = prim_key // 100` (the last two digits are the person within the household). Check: household sizes printed below are small (mostly 1–2, a handful up to ~9) and consistent with this, but it is an inference from the ID structure, not from a codebook.
- **State:** `state_code` = digits 2–3 of `prim_key` (the first two digits take only 4 values, so they are not a state). This gives 36 distinct values, matching the number of Indian states/UTs, but it is **presumed** — confirm against the LASI codebook before reporting it as "state".
- **Current smoking only:** `current_smoking` is only asked of ever-users. We assume an ever-user with no `current_smoking` answer did not smoke cigarettes/bidis (e.g. smokeless-only users).
- **Contradictions** (e.g. "never used tobacco" but "currently smoking") are resolved in favour of the more specific current-use answer and counted below.
- **BP/pulse plausibility bounds** (SBP 60–260, DBP 30–160, pulse 30–200 bpm, SBP must exceed DBP; BMI 12–50) are deliberately wide: they catch data-entry errors, they do not define "normal".
- Family-history code `7` = "None"; other digits = relatives (1 father … 6 grandchildren); `.`, `.d`, `.r` etc. = missing.
- Respondents with an unknown primary outcome (11) are kept here and dropped at the split (NB2).

## Limitations that this notebook cannot fix
- **The outcome is self-reported, physician-diagnosed, and prevalent** (cross-sectional): people already diagnosed may have changed BP, weight, smoking, activity. Predictors were measured *after* diagnosis, so associations may reflect reverse causation.
- **No survey weights, strata or PSUs** are in the merged file, so nothing here is nationally representative. Prevalences are unweighted sample proportions.
- No laboratory data (lipids, glucose/HbA1c), and no medication data (e.g. antihypertensives) are used.
- `ihd_composite` is only defined through questions asked of people who said "Yes" to chronic heart disease; heart attacks in people who answered "No" to `ht006` cannot be seen.

## 0. Environment setup

In [1]:
# ============================================================
# ENVIRONMENT SETUP (same logic as before + one optional override)
# ============================================================
import os

WINDOWS_PATH = r"C:\capstone\capstone_exec\data"
COLAB_DRIVE_PATH = "/content/drive/MyDrive/capstone_exec/data"

# Optional: set the environment variable CAPSTONE_DATA to force a data folder
# (used only so the notebooks can be executed headlessly; ignore it on your own machine).
if os.environ.get("CAPSTONE_DATA"):
    BASE_PATH = os.environ["CAPSTONE_DATA"]
    print("Using CAPSTONE_DATA override.")
elif os.path.exists("/content"):
    print("Detected a genuine hosted Colab runtime (found /content).")
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_PATH = COLAB_DRIVE_PATH
else:
    print("No /content found - treating this as a local machine (VS Code, Jupyter, etc).")
    BASE_PATH = WINDOWS_PATH

if not os.path.exists(BASE_PATH):
    fallback_candidates = [
        r"C:\capstone\capstone_exec\data",
        r"C:\capstone\possible datasets\lasi-1",
        "/content/drive/MyDrive/capstone_exec/data",
        os.path.join(os.getcwd(), "data"),
    ]
    found = next((c for c in fallback_candidates if os.path.exists(c)), None)
    if found:
        print(f"BASE_PATH did not exist, using fallback: {found}")
        BASE_PATH = found
    else:
        raise FileNotFoundError("Could not find the data folder anywhere. Checked:\n  "
                                + "\n  ".join([BASE_PATH] + fallback_candidates))

LASI_RAW = os.path.join(BASE_PATH, "lasi", "raw")
LASI_PROCESSED = os.path.join(BASE_PATH, "lasi", "processed")
MODELS_DIR = os.path.join(BASE_PATH, "..", "outputs", "models")
FIGURES_DIR = os.path.join(BASE_PATH, "..", "outputs", "figures")
TABLES_DIR = os.path.join(BASE_PATH, "..", "outputs", "tables")
for _d in (LASI_PROCESSED, MODELS_DIR, FIGURES_DIR, TABLES_DIR):
    os.makedirs(_d, exist_ok=True)

print("\nBASE_PATH:", BASE_PATH)
print("LASI_PROCESSED:", LASI_PROCESSED)

No /content found - treating this as a local machine (VS Code, Jupyter, etc).

BASE_PATH: C:\capstone\capstone_exec\data
LASI_PROCESSED: C:\capstone\capstone_exec\data\lasi\processed


## 1. Load the merged data
Duplicated `prim_key` values would mean one person appears twice, so we check.

In [2]:
import pandas as pd, numpy as np

INPUT_FILE = os.path.join(LASI_RAW, "lasi_merged_clear.csv")
df = pd.read_csv(INPUT_FILE)
print("Loaded shape:", df.shape)
print("Duplicate prim_key (must be 0):", df["prim_key"].duplicated().sum())

Loaded shape: (66859, 54)
Duplicate prim_key (must be 0): 0


## 2. Household and state identifiers
`household_id` is used to keep members of one household on the same side of the split (family members share environment, diet, and often reported family history, so a random person-level split is optimistic). `state_code` is used for leave-states-out validation.

In [3]:
pk = df["prim_key"].astype(str)
assert pk.str.len().eq(15).all(), "prim_key is expected to be 15 digits everywhere"

df["household_id"] = df["prim_key"] // 100
df["state_code"] = pk.str[1:3]

print("Number of households:", df["household_id"].nunique())
print("Household size distribution (people per household):")
print(df.groupby("household_id").size().value_counts().sort_index().to_string())
print("\nDistinct first-two-digit prefixes (NOT a state, only 4 values):", sorted(pk.str[:2].unique()))
print("Distinct state_code values (digits 2-3):", df["state_code"].nunique())
print(sorted(df["state_code"].unique()))

Number of households: 40419
Household size distribution (people per household):
1    17145
2    20668
3     2140
4      391
5       62
6       10
7        1
8        1
9        1

Distinct first-two-digit prefixes (NOT a state, only 4 values): ['10', '11', '12', '13']
Distinct state_code values (digits 2-3): 36
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36']


### How to read this output
Household sizes should be small (mostly 1–2, a handful up to ~9). `state_code` should have roughly 35–36 distinct values. If this is not the case the ID-structure assumption is wrong and both the household-grouped split and the state validation should be rechecked against the codebook.

## 3. Tobacco: Never / Former / Current (fixes the main coding bug)
Rules (applied in this order; later rules win):
1. `Never` — `ever_tobacco_use == "No"`.
2. `Former` — answered "Quit smoking" or "Quit smokeless tobacco" and is not a current user.
3. `Current` — currently smoking **or** currently using smokeless tobacco.
4. Anything else stays missing (e.g. ever-user with no current/quit answer, or no answer to the ever-use question).

`current_tobacco_any` (Yes/No) keeps the old binary meaning *but now "No" includes former users*. `current_smoking_only` is the stricter smoking definition used for the WHO chart sensitivity analysis.

In [4]:
cur_smoke = df["current_smoking"].eq("Currently smoking")
cur_smokeless = df["current_smokeless_tobacco"].eq("Currently using smokeless tobacco")
cur_any = cur_smoke | cur_smokeless
quit_any = df["current_smoking"].eq("Quit smoking") | df["current_smokeless_tobacco"].eq("Quit smokeless tobacco")
never = df["ever_tobacco_use"].eq("No")

# contradictions, counted before they are resolved
print("'Never used tobacco' but currently using :", int((never & cur_any).sum()))
print("'Never used tobacco' but answered 'quit' :", int((never & quit_any & ~cur_any).sum()))

status = np.select(
    [cur_any, quit_any, never],
    ["Current", "Former", "Never"],
    default="UNK",
).astype(object)
status[status == "UNK"] = np.nan
df["tobacco_status"] = status

df["current_tobacco_any"] = df["tobacco_status"].map({"Current": "Yes", "Former": "No", "Never": "No"})

# stricter smoking-only flag (assumes ever-users with no current_smoking answer did not smoke)
ever_known = df["ever_tobacco_use"].notna()
smk = np.select([cur_smoke, ever_known], ["Yes", "No"], default="UNK").astype(object)
smk[smk == "UNK"] = np.nan
df["current_smoking_only"] = smk

print("\ntobacco_status:")
print(df["tobacco_status"].value_counts(dropna=False).to_string())
print("\ncurrent_tobacco_any:")
print(df["current_tobacco_any"].value_counts(dropna=False).to_string())
print("\ncurrent_smoking_only:")
print(df["current_smoking_only"].value_counts(dropna=False).to_string())

# the check that exposed the original bug: disease prevalence by tobacco status
chk = df.dropna(subset=["chronic_heart_disease"]).groupby(df["tobacco_status"].fillna("Unknown"))["chronic_heart_disease"] \
        .agg(n="size", prevalence=lambda s: (s == "Yes").mean())
print("\nPrevalence of chronic heart disease by tobacco status (unweighted):")
print(chk.round(4).to_string())

'Never used tobacco' but currently using : 6
'Never used tobacco' but answered 'quit' : 28

tobacco_status:
tobacco_status
Never      43649
Current    19709
Former      3412
NaN           89

current_tobacco_any:
current_tobacco_any
No     47061
Yes    19709
NaN       89

current_smoking_only:
current_smoking_only
No     58351
Yes     8422
NaN       86

Prevalence of chronic heart disease by tobacco status (unweighted):
                    n  prevalence
tobacco_status                   
Current         19706      0.0273
Former           3412      0.0838
Never           43643      0.0329
Unknown            87      0.0345


### How to read this output
- "Unknown" should now be tiny (tens, not thousands). The old version had ~3,473 "missing", of which ~3,384 were really former users.
- Former users usually show the **highest** prevalence. This is *not* evidence that quitting causes heart disease; the most likely explanation is that many people quit **after** being diagnosed (reverse causation). Keep that in mind whenever the `Former` coefficient is interpreted (NB4) — it is a marker of "already ill", not a risk factor.
- "Current" users often show *lower* prevalence than "Never" users in this cross-sectional sample; again this reflects survivorship, sex mix (tobacco use is more common in men) and age structure, not a protective effect.

## 4. Family history (unknown stays unknown)
Each of `fm303`–`fm306` is a dash-separated multi-select string. We create a Yes/No summary flag per condition. **"Don't know"/refused codes stay missing** (the old code silently converted them to "No").

In [5]:
RELATIVE_MAP = {"1": "father", "2": "mother", "3": "brother", "4": "sister", "5": "children", "6": "grandchildren"}
MISSING_CODES = {"", ".", ".d", ".r", ".m", ".e", "nan"}

def any_relative(raw):
    """1.0 = at least one relative, 0.0 = 'None' (code 7), NaN = unknown/refused/missing."""
    if pd.isna(raw):
        return np.nan
    raw = str(raw).strip()
    if raw in MISSING_CODES:
        return np.nan
    if raw == "7":
        return 0.0
    codes = [c for c in raw.split("-") if c in RELATIVE_MAP]
    return 1.0 if codes else np.nan   # an unparseable code is treated as unknown, not as 'No'

FAMILY_HISTORY_VARS = {"fm303": "hypertension", "fm304": "diabetes", "fm305": "heart_disease", "fm306": "stroke"}
for col, name in FAMILY_HISTORY_VARS.items():
    flag = df[col].apply(any_relative)
    df[f"family_history_{name}_any"] = flag.map({1.0: "Yes", 0.0: "No"})
    print(f"{col} -> family_history_{name}_any:", df[f"family_history_{name}_any"].value_counts(dropna=False).to_dict())

fm303 -> family_history_hypertension_any: {'No': 51161, 'Yes': 15480, nan: 218}
fm304 -> family_history_diabetes_any: {'No': 55850, 'Yes': 10788, nan: 221}
fm305 -> family_history_heart_disease_any: {'No': 61474, 'Yes': 5178, nan: 207}
fm306 -> family_history_stroke_any: {'No': 63341, 'Yes': 3326, nan: 192}


### How to read this output
Each flag should now show a small, non-zero `nan` count (the "don't know"/refused answers). The old version showed zero missing for all four, which was the visible symptom of the bug. Family history is also **self-reported by the respondent**, so it is subject to recall bias and to differential awareness (people with a diagnosis in the family tend to know it).

## 5. Physical-plausibility checks (BMI, BP, pulse)
Implausible values are set to missing (not dropped), so the person's other data is kept. Bounds are intentionally wide (see assumptions). BP/pulse values of exactly 0 are measurement/entry errors.

In [6]:
df["BMI"] = df["weight_kg"] / ((df["height_cm"] / 100) ** 2)

bounds = {"BMI": (12, 50), "average_systolic_bp_mmHg": (60, 260),
          "average_diastolic_bp_mmHg": (30, 160), "average_pulse_bpm": (30, 200)}
for col, (lo, hi) in bounds.items():
    bad = (df[col] < lo) | (df[col] > hi)
    print(f"{col:28s} outside [{lo}, {hi}]: {int(bad.sum()):4d}   (min={df[col].min():.1f}, max={df[col].max():.1f})")
    df.loc[bad, col] = np.nan

# SBP must exceed DBP; if not, both readings are unreliable
swap = df["average_systolic_bp_mmHg"] <= df["average_diastolic_bp_mmHg"]
print("\nSBP <= DBP (both set to missing):", int(swap.sum()))
df.loc[swap, ["average_systolic_bp_mmHg", "average_diastolic_bp_mmHg"]] = np.nan

print("\nAfter cleaning:")
print(df[list(bounds)].describe().round(1).to_string())

BMI                          outside [12, 50]:   44   (min=9.3, max=145.6)
average_systolic_bp_mmHg     outside [60, 260]:    1   (min=0.0, max=233.5)
average_diastolic_bp_mmHg    outside [30, 160]:    2   (min=0.0, max=163.0)
average_pulse_bpm            outside [30, 200]:    1   (min=0.0, max=150.0)

SBP <= DBP (both set to missing): 0

After cleaning:
           BMI  average_systolic_bp_mmHg  average_diastolic_bp_mmHg  average_pulse_bpm
count  66059.0                   66680.0                    66679.0            66680.0
mean      22.9                     128.2                       81.7               80.4
std        4.7                      19.2                       10.4               11.7
min       12.0                      60.0                       41.0               32.0
25%       19.5                     114.4                       74.5               72.5
50%       22.5                     126.0                       81.0               80.0
75%       25.8                    

### How to read this output
A few dozen implausible values in total is expected. The summary after cleaning should show sensible minima/maxima. Age is **not** capped: LASI includes respondents up to 116 years (probably top-coded or mis-reported at the extreme); this is left as is and noted as a limitation.

## 6. Outcomes
- **`chronic_heart_disease`** (PRIMARY): "Has a doctor ever diagnosed you with chronic heart disease?" — self-reported, prevalent, **not** specific to coronary artery disease.
- **`ihd_composite`** (SECONDARY / sensitivity): `Yes` if the person reports a heart attack **or** a coronary-disease/blockage diagnosis; `No` if they answered No to chronic heart disease, or Yes to it but No to both follow-ups; missing if the follow-ups are incomplete (e.g. heart attack "No" and coronary question unanswered).
- **`coronary_disease_fixed`**: the coronary follow-up with the skip-pattern fixed (people who answered "No" to chronic heart disease were never asked, so they are "No", not missing). Kept for reference; small n.

In [7]:
chd = df["chronic_heart_disease"]
ha = df["heart_attack"]
cor = df["coronary_disease_answer"]

df["coronary_disease_fixed"] = cor
df.loc[chd == "No", "coronary_disease_fixed"] = "No"

ihd = np.select(
    [chd.eq("No"),
     chd.eq("Yes") & (ha.eq("Yes") | cor.eq("Yes")),
     chd.eq("Yes") & ha.eq("No") & cor.eq("No")],
    ["No", "Yes", "No"],
    default="UNK",
).astype(object)
ihd[ihd == "UNK"] = np.nan
df["ihd_composite"] = ihd

print("chronic_heart_disease:", chd.value_counts(dropna=False).to_dict())
print("ihd_composite        :", df["ihd_composite"].value_counts(dropna=False).to_dict())
print("coronary_disease_fixed:", df["coronary_disease_fixed"].value_counts(dropna=False).to_dict())
print("\nAmong people with chronic heart disease = Yes:")
sub = df[chd == "Yes"]
print(pd.crosstab(sub["heart_attack"].fillna("NA"), sub["coronary_disease_answer"].fillna("NA"),
                  rownames=["heart_attack"], colnames=["coronary_answer"]).to_string())

chronic_heart_disease: {'No': 64583, 'Yes': 2265, nan: 11}
ihd_composite        : {'No': 65245, 'Yes': 1508, nan: 106}
coronary_disease_fixed: {'No': 66089, 'Yes': 610, nan: 160}

Among people with chronic heart disease = Yes:
coronary_answer  NA   No  Yes
heart_attack                 
NA                0    1    1
No               94  662  271
Yes              55  843  338


### How to read this output
`ihd_composite = Yes` should be well below the primary outcome count (heart failure, valvular and other non-ischaemic problems are excluded) but larger than the coronary-only count, because heart attacks are included. A person with `ihd_composite = No` who has chronic heart disease = Yes simply reports a non-coronary / non-heart-attack condition **or** does not know; the data cannot separate these.

## 7. Assemble the modelling table and save
No imputation happens here. The complete-case file keeps rows with no missing predictor and a known primary outcome (used for a sensitivity analysis in NB4).

In [8]:
ID_COLS = ["prim_key", "household_id", "state_code"]
CONTINUOUS = ["age_years", "average_systolic_bp_mmHg", "average_diastolic_bp_mmHg", "average_pulse_bpm", "BMI"]
CATEGORICAL = ["sex", "diabetes", "tobacco_status", "vigorous_activity_frequency", "moderate_activity_frequency",
               "family_history_hypertension_any", "family_history_diabetes_any",
               "family_history_heart_disease_any", "family_history_stroke_any"]
EXTRA = ["current_tobacco_any", "current_smoking_only"]   # used by the WHO benchmark only
OUTCOMES = ["chronic_heart_disease", "ihd_composite", "coronary_disease_fixed"]

model_df = df[ID_COLS + CONTINUOUS + CATEGORICAL + EXTRA + OUTCOMES].copy()
print("Final shape:", model_df.shape)
print("\nMissing values per column (these are now genuine missing, NOT imputed):")
print(model_df.isna().sum().sort_values(ascending=False).to_string())

complete_case = model_df.dropna(subset=CONTINUOUS + CATEGORICAL + ["chronic_heart_disease"]).copy()
print("\nComplete-case rows:", len(complete_case), f"({len(complete_case)/len(model_df):.1%} of all rows)")

PRIMARY_OUTPUT = os.path.join(LASI_PROCESSED, "lasi_model_ready.csv")
SECONDARY_OUTPUT = os.path.join(LASI_PROCESSED, "lasi_model_ready_complete_case.csv")
model_df.to_csv(PRIMARY_OUTPUT, index=False)
complete_case.to_csv(SECONDARY_OUTPUT, index=False)
print("\nSaved:", PRIMARY_OUTPUT)
print("Saved:", SECONDARY_OUTPUT)

Final shape: (66859, 22)

Missing values per column (these are now genuine missing, NOT imputed):
BMI                                 800
family_history_diabetes_any         221
family_history_hypertension_any     218
family_history_heart_disease_any    207
family_history_stroke_any           192
average_diastolic_bp_mmHg           180
average_systolic_bp_mmHg            179
average_pulse_bpm                   179
coronary_disease_fixed              160
ihd_composite                       106
tobacco_status                       89
current_tobacco_any                  89
vigorous_activity_frequency          87
current_smoking_only                 86
moderate_activity_frequency          75
diabetes                             20
chronic_heart_disease                11
household_id                          0
state_code                            0
age_years                             0
prim_key                              0
sex                                   0

Complete-case rows: 6

## Summary
- `lasi_model_ready.csv`: un-imputed, with 3-level tobacco, correct family-history missingness, cleaned vitals, household/state IDs and three outcome columns.
- Imputation (multiple-imputation-style `IterativeImputer` for continuous variables, an explicit `Unknown` level for categoricals) now happens **inside the model pipeline in NB3**.
- **Still true, and must be stated in the paper:** self-reported prevalent outcome, no survey weights, post-diagnosis predictors, tobacco/activity/family history self-reported.

**Next:** `nb2_lasi_train_test_split.ipynb`.